# Customer Churn Prediction Model
Trains an XGBoost classifier on `NICE_CALLCENTER_DB.FEATURE_STORE.CHURN_TRAINING_DATA` and registers it in the Snowflake Model Registry.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

In [ ]:
session = get_active_session()
df = session.table('NICE_CALLCENTER_DB.FEATURE_STORE.CHURN_TRAINING_DATA').to_pandas()
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
df.head()

In [ ]:
FEATURE_COLS = [
    'INTERACTION_COUNT', 'AVG_DURATION_SEC', 'AVG_WRAP_UP_SEC',
    'AVG_HANDLE_TIME_SEC', 'TRANSFER_COUNT', 'ESCALATION_COUNT',
    'AVG_SENTIMENT', 'AVG_CSAT', 'AVG_NPS', 'AVG_CES',
    'FCR_RATE', 'ACCOUNT_TYPE_RANK', 'CONTRACT_TIER_RANK'
]
TARGET_COL = 'IS_CHURNED'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

X = X.fillna(X.median())

print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
print(f"Churn rate: {y.mean():.2%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

In [ ]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
model.fit(X_train, y_train)
print("Model training complete.")

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy:  {acc:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

importances = model.feature_importances_
idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(range(len(idx)), importances[idx])
ax.set_yticks(range(len(idx)))
ax.set_yticklabels([FEATURE_COLS[i] for i in idx])
ax.set_xlabel('Feature Importance')
ax.set_title('Churn Model - Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
from snowflake.ml.registry import Registry

session.sql('USE DATABASE NICE_CALLCENTER_DB').collect()
session.sql('USE SCHEMA FEATURE_STORE').collect()

registry = Registry(session=session, database_name='NICE_CALLCENTER_DB', schema_name='FEATURE_STORE')

try:
    registry.delete_model('CUSTOMER_CHURN_MODEL')
except:
    pass

model_version = registry.log_model(
    model=model,
    model_name='CUSTOMER_CHURN_MODEL',
    version_name='V1',
    sample_input_data=X_train.head(10),
    target_platforms=['WAREHOUSE'],
    conda_dependencies=['xgboost', 'scikit-learn'],
    metrics={
        'accuracy': float(acc),
        'f1_score': float(f1),
        'roc_auc': float(auc)
    },
    comment='XGBoost churn classifier trained on call center feature store data'
)

print(f"Model registered: {model_version.model_name} / {model_version.version_name}")
print(f"Target: WAREHOUSE")
print(f"Metrics: accuracy={acc:.4f}, f1={f1:.4f}, auc={auc:.4f}")

In [ ]:
test_input = X_test.head(5)
preds_local = model.predict(test_input)
print("Sample predictions (from local model):")
for i, (idx, row) in enumerate(test_input.iterrows()):
    print(f"  Row {i}: Predicted={'Churned' if preds_local[i]==1 else 'Retained'}")

print(f"\nModel '{model_version.model_name}' version '{model_version.version_name}' is registered in NICE_CALLCENTER_DB.FEATURE_STORE")